In [19]:
import os

In [20]:
%pwd

'C:\\Users\\Amanda\\Desktop\\Text-Summarizer\\Text-Summarizer-Project'

In [21]:
os.chdir("C:/Users/Amanda/Desktop/Text-Summarizer/Text-Summarizer-Project")

In [22]:
%pwd

'C:\\Users\\Amanda\\Desktop\\Text-Summarizer\\Text-Summarizer-Project'

In [23]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str


In [24]:
%pip install python-box ensure pyYAML joblib
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "src")))
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

Note: you may need to restart the kernel to use updated packages.


In [25]:
class ConfiguartionManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name
        )

        return data_transformation_config

In [27]:
!pip install transformers datasets sentencepiece accelerate


import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset,load_from_disk


  Using cached datasets-4.4.2-py3-none-any.whl.metadata (19 kB)
Using cached datasets-4.4.2-py3-none-any.whl (512 kB)


c:\Users\Amanda\.conda\envs\textS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras 3.11.3 requires h5py, which is not installed.
keras 3.11.3 requires rich, which is not installed.
tensorboard 2.20.0 requires markdown>=2.6.8, which is not installed.
tensorboard 2.20.0 requires pillow, which is not installed.
tensorboard 2.20.0 requires protobuf!=4.24.0,>=3.19.6, which is not installed.
tensorboard 2.20.0 requires werkzeug>=1.0.1, which is not installed.
tensorflow 2.20.0 requires h5py>=3.11.0, which is not installed.
tensorflow 2.20.0 requires protobuf>=5.28.0, which is not installed.
tensorflow 2.20.0 requires wrapt>=1.11.0, which is no

  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached datasets-4.4.2-py3-none-any.whl.metadata (19 kB)
  Using cached sentencepiece-0.2.1-cp313-cp313-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2025.11.3-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.7

In [35]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(
            example_batch['dialogue'],
            max_length=1024,
            truncation=True,
            padding="max_length"
        )

        with self.tokenizer.as_target_tokenizer():
            target_encodings = self.tokenizer(
                example_batch['summary'],
                max_length=128,
                truncation=True,
                padding="max_length"
            )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)

        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True
        )

        dataset_samsum_pt.save_to_disk(
            os.path.join(self.config.root_dir, "samsum_dataset")
        )



In [41]:
!pip install protobuf==3.20.3

try:
    config = ConfiguartionManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()

except Exception as e:
    raise e

  Using cached protobuf-3.20.3-py2.py3-none-any.whl.metadata (720 bytes)
Using cached protobuf-3.20.3-py2.py3-none-any.whl (162 kB)
[2025-12-29 15:11:02,049]: INFO:common: yaml file: C:\Users\Amanda\Desktop\Text-Summarizer\Text-Summarizer-Project\config\config.yaml loaded successfully
[2025-12-29 15:11:02,051]: INFO:common: yaml file: C:\Users\Amanda\Desktop\Text-Summarizer\Text-Summarizer-Project\params.yaml loaded successfully
[2025-12-29 15:11:02,052]: INFO:common: created directory at: artifacts
[2025-12-29 15:11:02,053]: INFO:common: created directory at: artifacts/data_transformation


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorboard 2.20.0 requires markdown>=2.6.8, which is not installed.
tensorboard 2.20.0 requires pillow, which is not installed.
tensorboard 2.20.0 requires werkzeug>=1.0.1, which is not installed.
tensorflow 2.20.0 requires h5py>=3.11.0, which is not installed.
tensorflow 2.20.0 requires wrapt>=1.11.0, which is not installed.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 3.20.3 which is incompatible.
Map:   0%|          | 0/14732 [00:00<?, ? examples/s]c:\Users\Amanda\.conda\envs\textS\Lib\site-packages\transformers\tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use th